In [6]:
# Install library asli jika belum ada
!{sys.executable} -m pip  install vaderSentiment

import pandas as pd
import numpy as np
import os
import warnings
import sys
!{sys.executable} -m pip install vaderSentiment

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../../outputs/sentiment-analysis/VTB/data_translated.csv'
OUTPUT_DIR = '../../../outputs/sentiment-analysis/VTB'
NLTK_RESULT_PATH = '../../../outputs/sentiment-analysis/VTB/sentiment_vtb_final.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[INFO] Library vaderSentiment (Original) berhasil dimuat.")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[INFO] Library vaderSentiment (Original) berhasil dimuat.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
df = pd.read_csv(DATA_PATH)
df_clean = df.copy()

# Pastikan tidak ada NaN agar tidak error
df_clean['teks_translated'] = df_clean['teks_translated'].fillna('')

print(f"Data siap diproses: {len(df_clean)} tweet")

Data siap diproses: 13192 tweet


In [10]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Inisialisasi Analyzer dari library asli
analyzer = SentimentIntensityAnalyzer()

print("\n[PROSES] Menghitung skor menggunakan library vaderSentiment asli...")

# Fungsi scoring
def get_original_vader_scores(text):
    return analyzer.polarity_scores(text)

# Eksekusi
scores_list = df_clean['teks_translated'].apply(get_original_vader_scores)

# Ekstraksi skor
df_clean['compound_orig'] = scores_list.apply(lambda x: x['compound'])
df_clean['pos_orig'] = scores_list.apply(lambda x: x['pos'])
df_clean['neu_orig'] = scores_list.apply(lambda x: x['neu'])
df_clean['neg_orig'] = scores_list.apply(lambda x: x['neg'])

print("[INFO] Scoring dengan library asli selesai.")


[PROSES] Menghitung skor menggunakan library vaderSentiment asli...
[INFO] Scoring dengan library asli selesai.


In [11]:
print("\n=== VALIDATION CHECK: NLTK vs ORIGINAL LIB ===")

if os.path.exists(NLTK_RESULT_PATH):
    # Load hasil dari notebook NLTK sebelumnya
    df_nltk = pd.read_csv(NLTK_RESULT_PATH)
    
    # Gabungkan berdasarkan kolom 'no'
    df_compare = pd.merge(
        df_nltk[['no', 'compound_score']], 
        df_clean[['no', 'compound_orig']], 
        on='no'
    )
    
    # Hitung perbedaan (toleransi float sangat kecil 1e-9)
    df_compare['diff'] = np.abs(df_compare['compound_score'] - df_compare['compound_orig'])
    mismatch = df_compare[df_compare['diff'] > 1e-9]
    
    if len(mismatch) == 0:
        print(" HASIL IDENTIK: Tidak ada perbedaan skor antara NLTK dan library asli.")
        print(f"   Total data divalidasi: {len(df_compare)}")
    else:
        print(f" PERBEDAAN DITEMUKAN: Ada {len(mismatch)} data dengan skor berbeda.")
        print(mismatch.head())
else:
    print("[SKIP] File hasil NLTK tidak ditemukan. Pastikan 03_vader_scoring.ipynb sudah dijalankan.")


=== VALIDATION CHECK: NLTK vs ORIGINAL LIB ===
[SKIP] File hasil NLTK tidak ditemukan. Pastikan 03_vader_scoring.ipynb sudah dijalankan.


In [12]:
# Simpan hasil untuk dokumentasi tambahan
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sentiment_vtb_original_lib.csv')
df_clean.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Hasil scoring library asli disimpan di: {OUTPUT_FILE}")


[OUTPUT] Hasil scoring library asli disimpan di: ../../../outputs/sentiment-analysis/VTB\sentiment_vtb_original_lib.csv
